In [ ]:
# Example 1: public API surface — verify the primary exported names exist.
from opentelemetry.instrumentation.wsgi import (
    OpenTelemetryMiddleware,
    WSGIGetter,
    collect_request_attributes,
    wsgi_getter,
)

print(f"OpenTelemetryMiddleware: {OpenTelemetryMiddleware}")
print(f"WSGIGetter: {WSGIGetter}")
print(f"collect_request_attributes: {collect_request_attributes}")
print(f"wsgi_getter (instance): {wsgi_getter}")

assert callable(OpenTelemetryMiddleware)
assert isinstance(wsgi_getter, WSGIGetter)
assert callable(collect_request_attributes)

In [ ]:
# Example 2: WSGIGetter extracts headers from WSGI environ dict.
from opentelemetry.instrumentation.wsgi import wsgi_getter

environ = {
    "HTTP_HOST": "example.com",
    "HTTP_USER_AGENT": "pytest/1.0",
    "HTTP_X_CUSTOM_HEADER": "hello",
    "REQUEST_METHOD": "GET",
    "PATH_INFO": "/items",
}

host = wsgi_getter.get(environ, "host")
user_agent = wsgi_getter.get(environ, "user-agent")
custom = wsgi_getter.get(environ, "x-custom-header")
missing = wsgi_getter.get(environ, "x-not-there")
keys = wsgi_getter.keys(environ)

print(f"host = {host}")
print(f"user-agent = {user_agent}")
print(f"x-custom-header = {custom}")
print(f"missing = {missing}")
print(f"keys (sample) = {keys[:5]}")

assert host == ["example.com"]
assert user_agent == ["pytest/1.0"]
assert missing is None

In [ ]:
# Example 3: collect_request_attributes derives OTel HTTP semantic attributes.
from opentelemetry.instrumentation.wsgi import collect_request_attributes

environ = {
    "REQUEST_METHOD": "POST",
    "PATH_INFO": "/api/items",
    "QUERY_STRING": "page=1",
    "SERVER_NAME": "example.com",
    "SERVER_PORT": "443",
    "HTTP_HOST": "example.com",
    "HTTP_USER_AGENT": "pytest/1.0",
    "wsgi.url_scheme": "https",
    "SERVER_PROTOCOL": "HTTP/1.1",
}

attrs = collect_request_attributes(environ)
for k, v in attrs.items():
    print(f"  {k} = {v}")

assert isinstance(attrs, dict)
assert len(attrs) > 0

In [ ]:
# Example 4: OpenTelemetryMiddleware produces a real SERVER span, captured in-memory.
# (This package ships no auto-instrumentor class — WSGI is a spec, not a library;
# the middleware IS the instrumentation surface.)
from io import BytesIO

from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.wsgi import OpenTelemetryMiddleware, get_default_span_name

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))

def app(environ, start_response):
    start_response("200 OK", [("Content-Type", "text/plain")])
    return [b"ok"]

wrapped = OpenTelemetryMiddleware(app, tracer_provider=provider)
environ = {
    "REQUEST_METHOD": "GET", "PATH_INFO": "/ping", "SERVER_NAME": "localhost",
    "SERVER_PORT": "80", "HTTP_HOST": "localhost", "wsgi.url_scheme": "http",
    "wsgi.input": BytesIO(b""), "SERVER_PROTOCOL": "HTTP/1.1",
}
iterable = wrapped(environ, lambda status, headers, exc_info=None: None)
body = b"".join(iterable)
if hasattr(iterable, "close"):
    iterable.close()  # WSGI contract: span ends when the response iterable is closed

spans = exporter.get_finished_spans()
print(f"expected span name = {get_default_span_name(environ)!r}")
print(f"captured = {[(s.name, str(s.kind)) for s in spans]}")
assert body == b"ok"
assert len(spans) == 1
assert spans[0].name == get_default_span_name(environ)

In [ ]:
# Example 5: wrap a tiny WSGI app with OpenTelemetryMiddleware and call it.
from io import BytesIO

from opentelemetry.instrumentation.wsgi import OpenTelemetryMiddleware


def simple_app(environ, start_response):
    start_response("200 OK", [("Content-Type", "text/plain")])
    return [b"hello world"]


wrapped = OpenTelemetryMiddleware(simple_app)

environ = {
    "REQUEST_METHOD": "GET",
    "PATH_INFO": "/",
    "SERVER_NAME": "localhost",
    "SERVER_PORT": "80",
    "HTTP_HOST": "localhost",
    "wsgi.url_scheme": "http",
    "wsgi.input": BytesIO(b""),
    "SERVER_PROTOCOL": "HTTP/1.1",
}
captured = {}

def start_response(status, headers, exc_info=None):
    captured["status"] = status
    captured["headers"] = headers

body = b"".join(wrapped(environ, start_response))
print(f"status = {captured['status']}")
print(f"body = {body!r}")

assert captured["status"] == "200 OK"
assert body == b"hello world"